# AI Device Health Intelligence System (ADHIS)

## Notebook: 06 - GenAI Explanation Engine (Rule-to-Text)
**Objective:** Convert model signals (battery prediction + anomalies + clusters) into user-friendly recommendations.  
**Datasets:** 
- `data/processed/telemetry_with_anomalies.csv`
- `data/processed/telemetry_with_clusters.csv`

### What this notebook covers
- Merge anomaly + cluster outputs
- Generate natural-language explanations
- Produce recommendation fields for UI display

In [1]:
import pandas as pd
from pathlib import Path

anom = pd.read_csv("../data/processed/telemetry_with_anomalies.csv")
clu = pd.read_csv("../data/processed/telemetry_with_clusters.csv")

df = anom.merge(clu[["device_id", "cluster_name"]], on="device_id", how="left")
df.head()

,device_id,battery_cycles,avg_temp,screen_on_time,fast_charging_count,cpu_usage,battery_health,anomaly_flag,cluster_name
0,1,152,39.943346,6.350490,61,47.558977,84.217729,1,Moderate Users
1,2,485,28.242771,2.136752,118,42.799247,86.421857,1,Moderate Users
2,3,910,38.351099,8.193086,263,54.995024,59.240169,1,Battery-Risk
3,4,320,38.384832,1.973999,238,37.287270,77.761681,1,Moderate Users
4,5,156,23.632519,2.260195,254,36.969037,91.372518,1,Moderate Users


In [2]:
def explain_device(row):
    msgs = []

    # battery health
    if row["battery_health"] < 70:
        msgs.append("Battery health is low and may degrade further without optimization.")
    elif row["battery_health"] < 80:
        msgs.append("Battery health is moderate. Small changes can improve longevity.")
    else:
        msgs.append("Battery health looks stable.")

    # anomaly / overheating
    if row.get("anomaly_flag", 1) == -1 and row["avg_temp"] > 42:
        msgs.append("Overheating risk detected. Background activity and charging habits may be contributing.")

    # cluster behavior
    if row.get("cluster_name") == "Heavy Users":
        msgs.append("Usage pattern indicates heavy daily load. Consider reducing background apps and enabling optimized charging.")
    elif row.get("cluster_name") == "Battery-Risk":
        msgs.append("Your usage indicates higher battery stress. Reducing fast charging and heat exposure can help.")

    return " ".join(msgs)

df["genai_explanation"] = df.apply(explain_device, axis=1)
df[["device_id", "battery_health", "avg_temp", "cluster_name", "genai_explanation"]].head(10)

,device_id,battery_health,avg_temp,cluster_name,genai_explanation
0,1,84.217729,39.943346,Moderate Users,Battery health looks stable.
1,2,86.421857,28.242771,Moderate Users,Battery health looks stable.
2,3,59.240169,38.351099,Battery-Risk,Battery health is low and may degrade further ...
3,4,77.761681,38.384832,Moderate Users,Battery health is moderate. Small changes can ...
4,5,91.372518,23.632519,Moderate Users,Battery health looks stable.
5,6,91.609676,30.318557,Moderate Users,Battery health looks stable.
6,7,61.856279,33.020856,Battery-Risk,Battery health is low and may degrade further ...
7,8,90.271300,34.101880,Moderate Users,Battery health looks stable.
8,9,74.676946,31.284329,Heavy Users,Battery health is moderate. Small changes can ...
9,10,85.877022,31.014986,Moderate Users,Battery health looks stable.


In [3]:
OUT_DIR = Path("../data/processed")
df.to_csv(OUT_DIR / "telemetry_with_explanations.csv", index=False)
print("Saved:", OUT_DIR / "telemetry_with_explanations.csv")

Saved: ../data/processed/telemetry_with_explanations.csv
